# Inference

> For sleep stages!

In [ ]:
#| default_exp inference

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, torch, numpy as np, warnings, edfio
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sleepjepa.jepa import JEPASimpleLightning
from sleepjepa.heads import RNNProbingHead
from sleepjepa.train import PatchTFTSleepStage

from sleepjepa.signal import iir_filter, iqr_normalization, resample_waveform

import sleepjepa.jepa as sj
import sleepjepa
import sys

sys.modules['timeflies'] = sleepjepa # sleepjepa was renamed from timeflies 
sj.loss_pred = sj.jepa_mse_loss # loss functon was renamed
sys.modules["timeflies.jepa"] = sj

from pathlib import Path
import json

CLASSIFIER_HEAD_DEFAULTS = dict(c_in=7, 
                input_size=384,
                hidden_size=384,
                predict_every_n_patches=10, 
                n_classes=5, 
                num_rnn_layers=2,
                rnn_dropout=0.1, 
                module='GRU',
                bidirectional=True,
                affine=True,
                pool='average', 
                pre_norm=False, 
                mlp_final_head=True,
                linear_dropout=0.)

FREQUENCY_DEFAULT = 128
HYPNOGRAM_FREQUENCY_DEFAULT = 1
HYPNOGRAM_EPOCH_SECONDS_DEFAULT = 30
MAX_SEQUENCE_LENGTH_SECONDS_DEFAULT = (12*3600) # 12 hrs
MIN_SEQUENCE_LENGTH_SECONDS_DEFAULT = (6*3600) # 6 hrs
HYPNOGRAM_PADDING_DEFAULT = -100

CHANNELS_DEFAULT = ['ECG', 'EOG(L)', 'EMG', 'EEG', 'SaO2', 'THOR RES', 'ABDO RES']
FREQUENCY_FILTERS_DEFAULT = {'ECG': (None, 0.3),  # ecg
                     'EOG(L)': (0.3, 45), # EOG
                     'EMG': (None, 10), # EMG
                     'EEG': (0.3, 45), # EEG
                     'SaO2': (), # SpO2
                     'THOR RES': (0.1, 15), # Thor res 
                     'ABDO RES': (0.1, 15)} # abdo res

/opt/miniconda3/envs/sleepjepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#| export
def load_sleepjepa_models(models_dir, # The directory of the saved models
                         encoder_model_name, # the name of the encoder model
                         classifier_model_name, # the name of the classifier model
                         classifier_head_defaults=CLASSIFIER_HEAD_DEFAULTS, # the defaults for the classifier head, DO NOT CHANGE!
                        ):
    """
    Loads the sleepjepa models from the models directory

    Args:
        models_dir (str): The directory of the saved models
        encoder_model_name (str): The name of the encoder model
        classifier_model_name (str): The name of the classifier model
        classifier_head_defaults (dict): The defaults for the classifier head, DO NOT CHANGE!
    Returns:
        encoder (PatchTFTSimpleLightning): The encoder model
        ss_classifier (PatchTFTSleepStage): The classifier model
    """
    encoder = JEPASimpleLightning.load_from_checkpoint(os.path.join(models_dir, encoder_model_name), map_location='cpu')
    lp_model = RNNProbingHead(**classifier_head_defaults)
    ss_classifier = PatchTFTSleepStage.load_from_checkpoint(os.path.join(models_dir, classifier_model_name),
                                                            preloaded_model=encoder,
                                                            map_location='cpu',
                                                            metrics={}, # remove metrics from ckpt (require cuda and not needed)
                                                            linear_probing_head = lp_model
                                                        )
    return encoder, ss_classifier

In [ ]:
#| export
def download_sleepjepa_models(write_dir='', # The directory to write the models to
                             token=None # Your hugging face token to use to download the models
                             ):
    """
    Function to download SleepJEPA models from hugging face

    Args:
        write_dir (str): The directory to write the models to
        token (str): Your hugging face token to use to download the models
    """
    hf_hub_download(repo_id="benmfox/SleepJEPA", local_dir=write_dir, filename="sleepjepa_encoder.ckpt", token=token)
    hf_hub_download(repo_id="benmfox/SleepJEPA", local_dir=write_dir, filename="sleepjepa_stage_classifier.ckpt", token=token)

In [ ]:
#| export
def process_edf(edf_file_path, # The edf file path to perform inference on
                channels, # the channels to read from the edf file
                spo2_channel_name='SaO2', # the name of the spo2 channel, for resampling function
                reference_channels_dict={}, # the reference channels to subtract from the channels. The keys are the channels to subtract from, and the values are the reference channels.
                frequency=FREQUENCY_DEFAULT, # the frequency to resample the channels to. Do not change this!
                frequency_filters=FREQUENCY_FILTERS_DEFAULT, # the frequency filters to apply to the channels. Do not change this!
                max_sequence_length=MAX_SEQUENCE_LENGTH_SECONDS_DEFAULT,
                min_sequence_length=MIN_SEQUENCE_LENGTH_SECONDS_DEFAULT,
                verbose=True, # whether to print the verbose output.
                ):
    """
    Process the edf file to prepare it for inference. 
    This function is used to prepare the edf file for inference by reading the channels, resampling them to the correct frequency, filtering them, and padding them to the correct length.
    Do not change the default parameters (frequency, max_sequence_length, min_sequence_length, frequency_filters_ordered) of this function!

    Args:
        edf_file_path (str): The path to the edf file to perform inference on
        channels (list): The channels to read from the edf file
        spo2_channel_name: The name of the SpO2 channel. SpO2 is normalized differently.
        reference_channels_dict (dict): The reference channels to subtract from the channels. The keys are the channels to subtract from, and the values are the reference channels.
        frequency (int): The frequency to resample the channels (the key) to. 
        frequency_filters (dict): The frequency filters to apply to the channels. 
        max_sequence_length (int): The maximum length of the sequence in seconds. 
        min_sequence_length (int): The minimum length of the sequence in seconds.
        verbose (bool): Whether to print the verbose output.

    Returns:
        signals (torch.Tensor): The processed signals
    """
    assert set(reference_channels_dict.keys()).issubset(set(channels)), 'The reference channels must be a subset of the channels'
    if len(set(channels) - set(frequency_filters.keys())) != 0:
        warnings.warn(f'`Channels` contain channels that will not be filtered: {set(channels) - set(frequency_filters.keys())}. If `dummy` channels, you can ignore this warning.')
    if frequency != FREQUENCY_DEFAULT:
        warnings.warn(f'Frequency is not set to the default of {FREQUENCY_DEFAULT} Hz. This will likely cause issues with the model.')
    f = edfio.read_edf(edf_file_path, lazy_load_data=True)
    num_channels = f.num_signals
    available_channels = [channel.upper() for channel in f.labels]
    dummy_idxs = [i for i,channel in enumerate(channels) if channel.lower() == 'dummy']
    channels = [channel.upper() for channel in channels if channel.lower() != 'dummy']
    if len(set(channels) - set(available_channels)) != 0:
        raise ValueError(f'Missing channels in edf file: {set(channels) - set(available_channels)}. Available channels: {available_channels}')
    channels_idxs = [available_channels.index(c) for c in channels]
    if reference_channels_dict:
        reference_channels = [channel.upper() for channel in reference_channels_dict.values() if channel is not None]
        if len(set(reference_channels) - set(available_channels)) != 0:
            raise ValueError(f'Missing reference channels in edf file: {set(reference_channels) - set(available_channels)}')
        reference_channels_idxs = [available_channels.index(c) for c in reference_channels]
    duration = int(f.duration)
   
    signal_headers = [{'label':f.signals[i].label, 
                        'dimension':f.signals[i].physical_dimension,
                        'sample_rate':f.signals[i].sampling_frequency,
                        'sample_frequency':f.signals[i].sampling_frequency,
                        'physical_max': f.signals[i].physical_max,
                        'physical_min': f.signals[i].physical_min,
                        'digital_max': f.signals[i].digital_max,
                        'digital_min': f.signals[i].digital_min,
                        'prefilter': f.signals[i].prefiltering,
                        'transducer': f.signals[i].transducer_type} for i in channels_idxs]
    if duration == -1:
        warnings.warn(f"Duration is -1 for file {edf_file_path}. Inferring from signal...")
        test_channel = f.signals[channels_idxs[0]].data
        signal_frequency = signal_headers[0].get('sample_frequency', None)
        if signal_frequency is None:
            duration = len(test_channel) // frequency
        else:
            duration = len(test_channel) // signal_frequency
    elif duration < min_sequence_length:
        warnings.warn(f'Duration of edf file is less than the minimum expected ({min_sequence_length} seconds): {duration}')
    elif duration > max_sequence_length:
        warnings.warn(f'Duration of edf file is greater than the maximum expected ({max_sequence_length} seconds): {duration}')

    if verbose:
        print(f'EDF file: {edf_file_path}')
        print(f'Duration: {duration}')
        print(f'Number of channels: {num_channels}')
        print(f'Available channels: {available_channels}')
        print(f'Signal headers: {signal_headers}')
    signals = []
    required_length = duration * int(frequency)
    for c_idx, channel_name in zip(channels_idxs, channels):
        signal = f.signals[c_idx].data
        channel_frequency = signal_headers[channels.index(channel_name)].get('sample_frequency', frequency)
        if reference_channels_dict and channel_name in reference_channels_dict and len(reference_channels_idxs) > 0:
            reference_signal = f.signals[reference_channels_idxs[reference_channels.index(channel_name)]].data
            signal = signal - reference_signal
        if len(signal) != required_length:
            signal = resample_waveform(signal, int(channel_frequency), int(frequency), is_spo2=channel_name == spo2_channel_name.upper())
        if channel_name in frequency_filters:
            freq_range = frequency_filters[channel_name]
            btype = 'highpass' if freq_range[0] is None else 'lowpass' if freq_range[1] is None else 'bandpass'
            freq_range = freq_range[1] if freq_range[0] is None else freq_range[0] if freq_range[1] is None else freq_range
            signal = iir_filter(signal, freq_range=freq_range, btype=btype, fs=frequency)
        signal = iqr_normalization(signal, is_spo2=channel_name == spo2_channel_name.upper())
        signals.append(signal)
    
    if len(dummy_idxs) > 0:
        for i in dummy_idxs:
            signals.insert(i, np.zeros(required_length))
    signals = np.array(signals, dtype=np.float32)
    assert signals.shape[0] == len(channels) + len(dummy_idxs), f"Signals shape is not equal to the number of channels: {signals.shape[0]} != {len(channels) + len(dummy_idxs)}"
    signals = torch.from_numpy(signals)
    return signals

In [ ]:
# #| notest
# edf_file_path = ''
# t = process_edf(edf_file_path,
#             channels=['dummy', 'EOG(L)', 'dummy','EEG','SaO2','THOR RES','ABDO RES'], 
#             spo2_channel_name='SaO2', # the name of the spo2 channel, for resampling function
#             reference_channels_dict={}, # the reference channels to subtract from the channels. The keys are the channels to subtract from, and the values are the reference channels.
#             frequency=FREQUENCY_DEFAULT, # the frequency to resample the channels to. Do not change this!
#             frequency_filters=FREQUENCY_FILTERS_DEFAULT, # the frequency filters to apply to the channels. Do not change this!
#             verbose=True)

In [ ]:
#| export
def view_edf_channels(edf_file_path, # The path to the edf file to view the channels of
                      uppercase=True # Whether to return the channels in uppercase
                      ):
    """
    View the channels of an edf file.

    Args:
        edf_file_path (str): The path to the edf file to view the channels of
        uppercase (bool): Whether to return the channels in uppercase

    Returns:
        channels (list): The channels in the edf file
    """
    f = edfio.read_edf(edf_file_path, lazy_load_data=True)
    if uppercase:
        return [channel.upper() for channel in f.labels]
    else:
        return f.labels

In [ ]:
#| export
def inference_nested_tensor_collate(batch):
    """
    Collate function for variable length sequences using NestedTensor.
    
    Args:
        batch: List of tensors (X) where:
            X: Tensor of shape (channels, seq_len)
            
    Returns:
        X_nested: NestedTensor containing the batch of sequences
    """
    # Convert to nested tensor
    ## the transpose is to make the jagged dimension the first dimension (which is the sequence length)
    ## if batches all have the same sequence length, torch automatically makes the jagged dimension the first dimension
    X = [i.transpose(0,1) for i in batch]

    X_nested = torch.nested.as_nested_tensor(X, layout=torch.jagged)
    X_nested = X_nested.transpose(1,2) # transpose back to the original shape
    return X_nested

In [ ]:
#| export
class EDFDataset(Dataset):
    def __init__(self, 
                edf_file_paths, # The paths to the edf files to perform inference on
                eeg_channel, # the EEG channel name in the EDF. The model was trained with C4-M1 and C3-M2 referenced EEG channels. However, 
                left_eog_channel, # the left EOG channel name in the EDF. The model was trained with M2 referenced left EOG channels.
                chin_emg_channel, # the chin EMG channel name in the EDF. The model was trained with chin refenced (chin 2 or chin 3) EMG channels.
                ecg_channel, # the ECG channel name in the EDF. The model was trained with augmented lead 2 ecg channels
                spo2_channel, # the SpO2 channel name in the EDF.
                abdomen_rr_channel, # the abdomen RR channel name in the EDF. 
                thoracic_rr_channel, # the thoracic RR channel name in the EDF.
                eeg_reference_channel=None, # the EEG reference channel name in the EDF. The model was trained with C4-M1 and C3-M2 referenced EEG channels. This will reference the channels, if they havent already been referenced. 
                left_eog_reference_channel=None, # the left EOG reference channel name in the EDF. The model was trained with M2 referenced left EOG channels. This will reference the channels, if they havent already been referenced. 
                chin_emg_reference_channel=None, # the chin EMG reference channel name in the EDF. The model was trained with chin refenced (chin 2 or chin 3) EMG channels. This will reference the channels, if they havent already been referenced. 
                ecg_reference_channel=None, # the ECG reference channel name in the EDF. The model was trained with augmented lead 2 ecg channels. This will reference the channels, if they havent already been referenced. 
                **kwargs
                ):
        """
        A dataset class for performing inference on multiple edf files.

        Args:
            edf_file_paths (list): The paths to the edf files to perform inference on
            eeg_channel (str): The name of the EEG channel in the EDF
            left_eog_channel (str): The name of the left EOG channel in the EDF
            chin_emg_channel (str): The name of the chin EMG channel in the EDF
            ecg_channel (str): The name of the ECG channel in the EDF
            spo2_channel (str): The name of the SpO2 channel in the EDF
            abdomen_rr_channel (str): The name of the abdomen RR channel in the EDF
            thoracic_rr_channel (str): The name of the thoracic RR channel in the EDF
            eeg_reference_channel (str): The name of the EEG reference channel in the EDF
            left_eog_reference_channel (str): The name of the left EOG reference channel in the EDF
            chin_emg_reference_channel (str): The name of the chin EMG reference channel in the EDF
            ecg_reference_channel (str): The name of the ECG reference channel in the EDF
            **kwargs: Additional keyword arguments for process_edf function
        """
        self.edf_file_paths = edf_file_paths
        self.channels = [ecg_channel, left_eog_channel, chin_emg_channel, eeg_channel, spo2_channel, thoracic_rr_channel, abdomen_rr_channel]
        self.channels = [c if c is not None and c != 'dummy' else 'dummy' for c in self.channels]
        self.reference_channels_dict = {ecg_channel: ecg_reference_channel,
                                        left_eog_channel: left_eog_reference_channel,
                                        chin_emg_channel: chin_emg_reference_channel,
                                        eeg_channel: eeg_reference_channel}
        self.kwargs = kwargs

    def __len__(self):
        return len(self.edf_file_paths)

    def __getitem__(self, idx):
        signals = process_edf(self.edf_file_paths[idx],
                    channels=self.channels,
                    reference_channels_dict=self.reference_channels_dict,
                    verbose=False,
                    **self.kwargs)
        return signals


In [ ]:
# import yaml
# yaml_path = '../sleepjepa_sleep_stage_inference_config.yaml'
# with open(yaml_path, 'r') as f:
#     yaml_data = yaml.safe_load(f)

# frequency_filters = list(yaml_data['frequency_filters'].values())
# frequency_filters = dict(zip(yaml_data['channels'].values(), frequency_filters))
# process_edf_kwargs = {'frequency_filters': frequency_filters,
#                         'frequency': yaml_data['frequency'],
#                         'min_sequence_length': yaml_data['min_sequence_length_sec'],
#                         'max_sequence_length': yaml_data['max_sequence_length_sec'],
#                         'spo2_channel_name': yaml_data['channels']['spo2']
#                         }

In [ ]:
# ds = EDFDataset(edf_file_paths=[''],
# eeg_channel='EEG',
# left_eog_channel='EOG(L)',
# chin_emg_channel='EMG',
# ecg_channel='ECG',
# spo2_channel='SaO2',
# abdomen_rr_channel='ABDO RES',
# thoracic_rr_channel='THOR RES',
# **process_edf_kwargs
# )

# data_loader = DataLoader(ds, batch_size=yaml_data['batch_size'], shuffle=False, pin_memory=yaml_data['pin_memory'], persistent_workers=yaml_data['persistent_workers'], num_workers=yaml_data['num_workers'], collate_fn=inference_nested_tensor_collate)
# # preds = infer_on_edf_dataset(edf_dataloader=data_loader, 
# #                             device=yaml_data['device'],
# #                             models_dir=yaml_data['models_dir'],
# #                             encoder_model_name=yaml_data['encoder_model_name'],
# #                             classifier_model_name=yaml_data['classifier_model_name']
# #                             )

In [ ]:
#| export
def map_stage(stage):
    if stage == 4:
        return 5  # REM
    elif stage in (0, 1, 2, 3):
        return stage
    else:
        return -1  # Undefined
 
def create_hypjson(epochs):
    return {
        "header": {
            "study_date": "",
            "study_time": "",
            "study_id": "",
            "version": "6.0.1.24"
        },
        "Data": {
            "10sEpochs": epochs
        },
        "Legend": {
            "undefined": -1,
            "awake": 0,
            "stage1": 1,
            "stage2": 2,
            "stage3": 3,
            "stage4": 4,
            "REM": 5
        }
    }

def write_pred_to_hypjson(predictions, hypjson_path):
    """
    Function to write the predictions to a hypjson file.
    """
    out = torch.softmax(predictions, dim=0) # apply softmax to get probabilities
    out = out.argmax(0).cpu().numpy().astype(int).repeat(3) # repeat each epoch 3 times to get 10s epochs from 30s epochs
    hypjson_epochs = list(map(map_stage, out.tolist())) # map stages
    hyp_json = create_hypjson(hypjson_epochs)
    with open(hypjson_path, 'w') as out_file:
        json.dump(hyp_json, out_file, indent=4)
    print(f'Saved hypjson file to {hypjson_path}')

In [ ]:
#| export
def infer_on_edf(edf_file_path, # The edf file path to perform inference on
                eeg_channel, # the EEG channel name in the EDF. The model was trained with C4-M1 and C3-M2 referenced EEG channels. However, 
                left_eog_channel, # the left EOG channel name in the EDF. The model was trained with M2 referenced left EOG channels.
                chin_emg_channel, # the chin EMG channel name in the EDF. The model was trained with chin refenced (chin 2 or chin 3) EMG channels.
                ecg_channel, # the ECG channel name in the EDF. The model was trained with augmented lead 2 ecg channels
                spo2_channel, # the SpO2 channel name in the EDF.
                abdomen_rr_channel, # the abdomen RR channel name in the EDF. 
                thoracic_rr_channel, # the thoracic RR channel name in the EDF.
                eeg_reference_channel=None, # the EEG reference channel name in the EDF. The model was trained with C4-M1 and C3-M2 referenced EEG channels. This will reference the channels, if they havent already been referenced. 
                left_eog_reference_channel=None, # the left EOG reference channel name in the EDF. The model was trained with M2 referenced left EOG channels. This will reference the channels, if they havent already been referenced. 
                chin_emg_reference_channel=None, # the chin EMG reference channel name in the EDF. The model was trained with chin refenced (chin 2 or chin 3) EMG channels. This will reference the channels, if they havent already been referenced. 
                ecg_reference_channel=None, # the ECG reference channel name in the EDF. The model was trained with augmented lead 2 ecg channels. This will reference the channels, if they havent already been referenced. 
                models_dir='', # the directory of the saved models
                encoder_model_name='sleepjepa_encoder.ckpt', # the name of the encoder model
                classifier_model_name='sleepjepa_stage_classifier.ckpt', # the name of the classifier model
                device="cpu", # the device to run the model on
                **kwargs
                ):
    """
    Performs inference on a single edf file using the sleepjepa models. 
    If you specify a channel as None or 'dummy', the channel will be passed through as a zero vector. This allows you to use the model even if some channels are not present in the edf file.

    Args:
        edf_file_path (str): The path to the edf file to perform inference on
        eeg_channel (str): The name of the EEG channel in the EDF
        left_eog_channel (str): The name of the left EOG channel in the EDF
        chin_emg_channel (str): The name of the chin EMG channel in the EDF
        ecg_channel (str): The name of the ECG channel in the EDF
        spo2_channel (str): The name of the SpO2 channel in the EDF
        abdomen_rr_channel (str): The name of the abdomen RR channel in the EDF
        thoracic_rr_channel (str): The name of the thoracic RR channel in the EDF
        eeg_reference_channel (str): The name of the EEG reference channel in the EDF
        left_eog_reference_channel (str): The name of the left EOG reference channel in the EDF
        chin_emg_reference_channel (str): The name of the chin EMG reference channel in the EDF
        ecg_reference_channel (str): The name of the ECG reference channel in the EDF
        models_dir (str): The directory of the saved models
        encoder_model_name (str): The name of the encoder model
        classifier_model_name (str): The name of the classifier model
        device (str): The device to run the model on
        **kwargs: Additional keyword arguments for process_edf function

    Returns:
        out (torch.Tensor): The sleep stage logit outputs of the classifier for each sleep epoch in the edf file
    """
    if not os.path.exists(os.path.join(models_dir, encoder_model_name)):
        raise ValueError(f"Encoder model not found in {models_dir}")
    if not os.path.exists(os.path.join(models_dir, classifier_model_name)):
        raise ValueError(f"Classifier model not found in {models_dir}")
    if not os.path.exists(edf_file_path):
        raise ValueError(f"EDF file not found in {edf_file_path}")
    try:
        _, ss_classifier = load_sleepjepa_models(models_dir, encoder_model_name, classifier_model_name)
    except Exception as e:
        raise ValueError(f"Trouble loading models: {e}")
    
    try:
        channels = [ecg_channel, left_eog_channel, chin_emg_channel, eeg_channel, spo2_channel, thoracic_rr_channel, abdomen_rr_channel]
        channels = [c if c is not None and c != 'dummy' else 'dummy' for c in channels]
        signals = process_edf(edf_file_path,
                            channels=channels,
                            reference_channels_dict={ecg_channel: ecg_reference_channel,
                                                    left_eog_channel: left_eog_reference_channel,
                                                    chin_emg_channel: chin_emg_reference_channel,
                                                    eeg_channel: eeg_reference_channel},
                            **kwargs
                            )
    except Exception as e:
        raise ValueError(f"Trouble processing edf file: {e}")
    
    try:
        # fine-tuned sleep stage classifier, recommend using a GPU
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            device = torch.device(device)
        ss_classifier = ss_classifier.to(device)
        ss_classifier.eval()

        with torch.no_grad():
            signals = signals.unsqueeze(0).to(device)
            out = ss_classifier(signals)
            out = out.squeeze(0).cpu()
        return out
    except Exception as e:
        raise ValueError(f"Trouble inferring on edf file: {e}")

In [ ]:
# out = infer_on_edf(edf_file_path='',
#              eeg_channel='EEG', 
#              left_eog_channel='EOG(L)', 
#              chin_emg_channel='EMG', 
#              ecg_channel='ECG', 
#              spo2_channel='SaO2', 
#              abdomen_rr_channel='ABDO RES', 
#              thoracic_rr_channel='THOR RES',
#              device='cpu',
#              models_dir='../../Downloads/'
#              )

In [ ]:
#| export
def infer_on_edf_dataset(edf_dataloader, # the edf dataset to perform inference on
                        models_dir='', # the directory of the saved models
                        device=None, # the device to run the model on
                        autocast=False, # whether to use autocast, only use on CUDA, not on CPU
                        encoder_model_name='sleepjepa_encoder.ckpt', # the name of the encoder model
                        classifier_model_name='sleepjepa_stage_classifier.ckpt', # the name of the classifier model
                        ):
    """
    Performs inference on an EDFDataset.

    Args:
        edf_dataloader (Dataset or DataLoader): The dataset (for cpu) or dataloader (from EDFDataset) to perform inference on
        batch_size (int): The batch size to use for inference
        models_dir (str): The directory of the saved models
        device (str): The device to run the model on
        autocast (bool): Whether to use automatic mixed precision
        encoder_model_name (str): The name of the encoder model
        classifier_model_name (str): The name of the classifier model

    Returns:
        preds (list): The predicted sleep stage logits for each edf file
    """
    _, ss_classifier = load_sleepjepa_models(models_dir, encoder_model_name, classifier_model_name)
    preds = []
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)
    if autocast and device.type == 'cpu':
        autocast = False
        warnings.warn('Autocast is slow on CPU. Disabling...')
    ss_classifier = ss_classifier.to(device)
    ss_classifier.eval()
    with torch.no_grad():
        for batch in tqdm(edf_dataloader, desc="Predicting"):
            x = batch.unsqueeze(0) if batch.dim() == 2 else batch
            x = x.to(device)
            if autocast:
                with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                    pred = ss_classifier(x) # [bs, n_classes, pred_len_seconds]
            else:
                pred = ss_classifier(x) # [bs, n_classes, pred_len_seconds]
            preds.append(pred.cpu())
    return torch.cat(preds)

In [ ]:
# preds = infer_on_edf_dataset(edf_dataloader=ds, 
#                                 device=yaml_data['device'],
#                                 models_dir=yaml_data['models_dir'],
#                                 encoder_model_name=yaml_data['encoder_model_name'],
#                                 classifier_model_name=yaml_data['classifier_model_name']
#                                 )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()